Using Python 3.12.6 environment at: /usr/local
Audited 2 packages in 23ms
Note: you may need to restart the kernel to use updated packages.


In [13]:
@triton.jit
def sum_pass_kernel(x_ptr, partial_ptr, num_el, BLOCK_SIZE:tl.constexpr):
    
    
    pid = tl.program_id(0)
    block_start = pid * BLOCK_SIZE
    offs = block_start + tl.arange(0, BLOCK_SIZE)
    
    mask = offs < num_el
    values = tl.load(
        x_ptr + offs, mask=mask, other=0.0
    )
    
    result = tl.sum(values, axis=0)
    
    tl.store(partial_ptr+pid, result)

In [5]:
def triton_sum(x):
    assert x.is_cuda
    assert x.dtype == torch.float32
    assert x.ndim == 1
    assert x.is_contiguous()
    assert x.numel() > 0

    curr = x
    block_size = 1024

    while curr.numel() > 1:
        num_el = curr.numel()

        num_of_programs = triton.cdiv(num_el, block_size)

        partial = torch.empty(num_of_programs, device=x.device, dtype=x.dtype)

        sum_pass_kernel[(num_of_programs,)](
            curr,
            partial,
            num_el,
            BLOCK_SIZE=block_size)
        curr = partial

    return curr

In [9]:
@triton.jit
def max_pass_kernel(x_ptr, partial_ptr, num_el, BLOCK_SIZE:tl.constexpr):
    pid = tl.program_id(0)
    
    block_start = pid * BLOCK_SIZE
    offs = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offs < num_el
    
    values = tl.load(x_ptr+offs, mask=mask, other=-float("inf"))
    
    result = tl.max(values, axis=0)
    
    tl.store(partial_ptr+pid, result)

In [6]:
def triton_max(x):
    assert x.is_cuda
    assert x.dtype == torch.float32
    assert x.ndim==1
    assert x.is_contiguous()
    assert x.numel() > 0

    curr = x
    block_size = 1024

    while curr.numel() > 1:
        num_el = curr.numel()

        num_of_programs=triton.cdiv(num_el, block_size)

        partial = torch.empty(
            num_of_programs,
            device=x.device,
            dtype=x.dtype
        )
        max_pass_kernel[(num_of_programs,)](
            curr, partial,
            num_el, BLOCK_SIZE=block_size
        )
        curr = partial

    return curr

If we used: partial_ptr+offs above  
each program would try to write its one result into several positions  
Program 0 writes P0 to positions 0, 1, 2, 3  
Program 1 writes P1 to positions 4, 5, 6, 7  
The rule is:  
One output per input element → use offsets
One reduced output per program → use program_id

In [16]:
torch.manual_seed(0)

x = torch.rand(100_003, device="cuda", dtype=torch.float32)

torch_sum = torch.sum(x)
torch_max = torch.max(x)

tritonSum = triton_sum(x).squeeze()
tritonMax = triton_max(x).squeeze()

print("Sum")
print("PyTorch:", torch_sum.item())
print("Triton: ", tritonSum.item())

print("\nMaximum")
print("PyTorch:", torch_max.item())
print("Triton: ", tritonMax.item())

Sum
PyTorch: 49991.7421875
Triton:  49991.74609375

Maximum
PyTorch: 0.9999988675117493
Triton:  0.9999988675117493
